# V10 Q1 Defense Notebook: PMLB Mechanism + Statistics + Mitigation

Run this single notebook in Kaggle for the final Q1-defense experiments.

Kaggle settings: **Internet ON**, **Accelerator None**.

Optional input: attach your main raw result CSV if available, named like `raw_results_final_24datasets_9models.csv`.

Final output: `/kaggle/working/v10_q1_defense_outputs.zip`.


In [ ]:

# =========================
# CONFIG
# =========================
OUT_ROOT = "/kaggle/working/v10_q1_defense"
MAX_ROWS_PER_DATASET = 20000
SEEDS = [0, 1, 2, 3, 4]

PMLB_DATASETS = [
    "adult", "mushroom", "allhypo", "clean2", "dna", "hypothyroid", "kr_vs_kp",
    "magic", "nursery", "optdigits", "pendigits", "phoneme", "satimage",
    "segmentation", "spambase", "splice", "tic_tac_toe", "vehicle",
    "vowel", "waveform_40", "mfeat_fourier", "mfeat_karhunen",
    "mfeat_morphological", "mfeat_zernike"
]

MECH_NOISE_LEVELS = [0.1, 0.2, 0.3]
MECH_IMBALANCE_LEVELS = [0.7, 0.8, 0.9]

RUN_MITIGATION = True
MITIGATION_MODELS = ["LogisticRegression", "RandomForest", "XGBoost", "LightGBM"]
MITIGATION_METHODS = ["none", "class_weight", "random_oversample"]
MITIGATION_SETTINGS = [
    ("clean", 0.0, 0.0, "natural"),
    ("noise_only", 0.2, 0.0, "natural"),
    ("missing_only", 0.0, 0.2, "natural"),
    ("imbalance_only", 0.0, 0.0, 0.8),
    ("noise_imbalance", 0.2, 0.0, 0.8),
    ("all_three", 0.2, 0.2, 0.8),
]

print("Datasets:", len(PMLB_DATASETS))
print("Output:", OUT_ROOT)


In [ ]:

# =========================
# SETUP
# =========================
import os, sys, glob, json, time, warnings, shutil, subprocess
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

for pkg in ["pmlb", "statsmodels", "xgboost", "lightgbm"]:
    try:
        __import__(pkg)
    except Exception:
        print("Installing", pkg)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import numpy as np
import pandas as pd
from pmlb import fetch_data
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, matthews_corrcoef, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import statsmodels.formula.api as smf

Path(OUT_ROOT).mkdir(parents=True, exist_ok=True)
print("Setup complete")


In [ ]:

# =========================
# UTILITIES
# =========================
def ensure_dir(path):
    path = Path(path); path.mkdir(parents=True, exist_ok=True); return path

def save_row(row, path):
    path = Path(path)
    pd.DataFrame([row]).to_csv(path, mode="a", header=not path.exists(), index=False)

def clean_columns(df):
    df = df.copy(); df.columns = [str(c).strip().replace(" ", "_").replace("/", "_") for c in df.columns]; return df

def encode_target(y):
    le = LabelEncoder(); return pd.Series(le.fit_transform(pd.Series(y).astype(str))).reset_index(drop=True)

def stratified_cap(X, y, max_rows=20000, seed=123):
    if len(X) <= max_rows:
        return X.reset_index(drop=True), y.reset_index(drop=True), False
    try:
        Xs, _, ys, _ = train_test_split(X, y, train_size=max_rows, random_state=seed, stratify=y)
        return Xs.reset_index(drop=True), ys.reset_index(drop=True), True
    except Exception:
        rng = np.random.default_rng(seed); idx = rng.choice(np.arange(len(X)), size=max_rows, replace=False)
        return X.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True), True

def load_pmlb_dataset(name):
    df = fetch_data(name, local_cache_dir="/kaggle/working/pmlb_cache")
    df = clean_columns(df)
    target_col = "target" if "target" in df.columns else df.columns[-1]
    X = df.drop(columns=[target_col])
    y = encode_target(df[target_col])
    keep = [c for c in X.columns if X[c].nunique(dropna=True) > 1]
    X = X[keep].copy()
    X, y, capped = stratified_cap(X, y, MAX_ROWS_PER_DATASET, 123)
    meta = {"dataset_name":name, "n_rows":int(len(X)), "n_features":int(X.shape[1]), "n_classes":int(y.nunique()), "majority_ratio":float(y.value_counts(normalize=True).max()), "minority_count":int(y.value_counts().min()), "capped_20000":bool(capped)}
    return X, y, meta

def build_pmlb_manifest():
    out_dir = ensure_dir(Path(OUT_ROOT)/"data_manifest")
    rows, failed = [], []
    for name in PMLB_DATASETS:
        try:
            X, y, meta = load_pmlb_dataset(name)
            rows.append(meta); print("Loaded", name, X.shape, "classes", y.nunique())
        except Exception as e:
            failed.append({"dataset_name":name, "error":str(e)[:500]}); print("Failed", name, e)
    pd.DataFrame(rows).to_csv(out_dir/"pmlb_included_manifest_v10.csv", index=False)
    pd.DataFrame(failed).to_csv(out_dir/"pmlb_failed_manifest_v10.csv", index=False)
    display(pd.DataFrame(rows))
    return pd.DataFrame(rows)


In [ ]:

# =========================
# CORRUPTION FUNCTIONS
# =========================
def apply_label_noise(y, rate, seed):
    rng = np.random.default_rng(seed); y_arr = np.asarray(y).copy(); flip_mask = np.zeros(len(y_arr), dtype=bool)
    if rate <= 0: return pd.Series(y_arr), flip_mask
    classes = np.unique(y_arr); n_flip = int(round(rate * len(y_arr)))
    if n_flip <= 0: return pd.Series(y_arr), flip_mask
    flip_idx = rng.choice(np.arange(len(y_arr)), size=n_flip, replace=False)
    for i in flip_idx:
        choices = classes[classes != y_arr[i]]
        if len(choices): y_arr[i] = rng.choice(choices); flip_mask[i] = True
    return pd.Series(y_arr), flip_mask

def apply_imbalance_observed_labels(X, y_observed, target_majority_ratio, seed):
    # Majority is recomputed from observed labels after noise; nonmajority examples are downsampled.
    if str(target_majority_ratio) == "natural":
        idx = np.arange(len(X)); vc = pd.Series(y_observed).value_counts()
        return X.reset_index(drop=True), pd.Series(y_observed).reset_index(drop=True), idx, {"observed_majority_class":int(vc.idxmax()), "desired_nonmajority":int((pd.Series(y_observed)!=vc.idxmax()).sum()), "actual_nonmajority":int((pd.Series(y_observed)!=vc.idxmax()).sum()), "imbalance_feasible":True}
    rng = np.random.default_rng(seed); X = X.reset_index(drop=True); y = pd.Series(y_observed).reset_index(drop=True); target_ratio = float(target_majority_ratio)
    counts = y.value_counts(); majority_class = counts.idxmax(); majority_idx = y[y == majority_class].index.to_numpy(); non_idx_all = y[y != majority_class].index.to_numpy(); non_classes = [c for c in counts.index if c != majority_class]
    n_majority = len(majority_idx); desired_total = int(round(n_majority / target_ratio)); desired_non = max(desired_total - n_majority, len(non_classes)); desired_non = min(desired_non, len(non_idx_all))
    selected, remaining = [], desired_non
    for c in non_classes:
        c_idx = y[y == c].index.to_numpy()
        if len(c_idx) > 0 and remaining > 0: selected.append(rng.choice(c_idx)); remaining -= 1
    if remaining > 0:
        used = set(selected); avail = np.array([i for i in non_idx_all if i not in used])
        if len(avail): selected.extend(rng.choice(avail, size=min(remaining, len(avail)), replace=False).tolist())
    keep_idx = np.concatenate([majority_idx, np.array(selected, dtype=int)]); rng.shuffle(keep_idx)
    info = {"observed_majority_class":int(majority_class), "desired_nonmajority":int(desired_non), "actual_nonmajority":int(len(selected)), "imbalance_feasible":bool(desired_non <= len(non_idx_all))}
    return X.iloc[keep_idx].reset_index(drop=True), y.iloc[keep_idx].reset_index(drop=True), keep_idx, info

def apply_mcar_missing(X, rate, seed):
    if rate <= 0: return X.copy()
    rng = np.random.default_rng(seed); return X.mask(rng.random(X.shape) < rate)

def corrupt_train_test(X_train, y_train, X_test, y_test, noise, missing, imbalance, seed):
    X_train = X_train.reset_index(drop=True); y_train = pd.Series(y_train).reset_index(drop=True); X_test = X_test.reset_index(drop=True); y_test = pd.Series(y_test).reset_index(drop=True)
    y_noisy, flip_mask = apply_label_noise(y_train, noise, seed+11)
    X_sub, y_sub, keep_idx, imb_info = apply_imbalance_observed_labels(X_train, y_noisy, imbalance, seed+22)
    X_sub = apply_mcar_missing(X_sub, missing, seed+33); X_test = apply_mcar_missing(X_test, missing, seed+44)
    diag = {"train_rows_after_corruption":int(len(X_sub)), "n_label_flips_before_sampling":int(flip_mask.sum())}; diag.update(imb_info)
    return X_sub, pd.Series(y_sub).reset_index(drop=True), X_test, y_test, keep_idx, flip_mask, diag


In [ ]:

# =========================
# PMLB MECHANISM DIAGNOSTICS
# =========================
def run_pmlb_mechanism_diagnostics():
    out_dir = ensure_dir(Path(OUT_ROOT)/"mechanism_pmlb")
    rows = []
    for dataset_name in PMLB_DATASETS:
        print("Mechanism:", dataset_name)
        try:
            X, y, meta = load_pmlb_dataset(dataset_name)
        except Exception as e:
            print("Skip", dataset_name, e); continue
        n_classes = int(y.nunique()); task_type = "binary" if n_classes == 2 else "multiclass"
        for seed in SEEDS:
            try: Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
            except Exception: continue
            Xtr = Xtr.reset_index(drop=True); ytr = pd.Series(ytr).reset_index(drop=True)
            counts = ytr.value_counts(); min_cls = int(counts.idxmin()); maj_cls = int(counts.idxmax()); orig_min = int(counts.min()); orig_maj = int(counts.max())
            for noise in MECH_NOISE_LEVELS:
                y_noisy, flip = apply_label_noise(ytr, noise, seed+101)
                for imb in MECH_IMBALANCE_LEVELS:
                    Xsub, ysub_obs, keep_idx, info = apply_imbalance_observed_labels(Xtr, y_noisy, imb, seed+202)
                    keep_idx = np.asarray(keep_idx); true_ret = ytr.iloc[keep_idx].reset_index(drop=True); ret_flip = flip[keep_idx]
                    clean_min_ret = int((true_ret == min_cls).sum()); clean_maj_ret = int((true_ret == maj_cls).sum())
                    misl_ret = int(ret_flip.sum()); misl_min = int(((true_ret == min_cls) & ret_flip).sum()); misl_maj = int(((true_ret == maj_cls) & ret_flip).sum())
                    rows.append({"dataset_name":dataset_name, "task_type":task_type, "n_classes":n_classes, "seed":seed, "label_noise":noise, "imbalance_level":imb, "original_train_rows":int(len(ytr)), "original_minority_count":orig_min, "original_majority_count":orig_maj, "clean_minority_class":min_cls, "clean_majority_class":maj_cls, "observed_majority_class_after_noise":info["observed_majority_class"], "post_imbalance_rows":int(len(ysub_obs)), "clean_minority_retained_count":clean_min_ret, "clean_majority_retained_count":clean_maj_ret, "clean_minority_retention_rate":clean_min_ret/max(orig_min,1), "effective_clean_minority_ratio":clean_min_ret/max(len(ysub_obs),1), "mislabeled_retained_count":misl_ret, "mislabeled_retained_ratio":misl_ret/max(len(ysub_obs),1), "mislabeled_minority_retained_count":misl_min, "mislabeled_majority_retained_count":misl_maj, "mislabeled_minority_retained_ratio":misl_min/max(clean_min_ret,1), "observed_majority_ratio_after":float(pd.Series(ysub_obs).value_counts(normalize=True).max()), "observed_minority_count_after":int(pd.Series(ysub_obs).value_counts().min()), "imbalance_feasible":info["imbalance_feasible"]})
    df = pd.DataFrame(rows); df.to_csv(out_dir/"pmlb_mechanism_diagnostics.csv", index=False)
    summary = df.groupby(["label_noise","imbalance_level"])[["clean_minority_retention_rate","effective_clean_minority_ratio","mislabeled_retained_ratio","mislabeled_minority_retained_ratio"]].mean().reset_index()
    summary.to_csv(out_dir/"pmlb_mechanism_summary.csv", index=False)
    bm = df.groupby(["task_type","label_noise","imbalance_level"])[["clean_minority_retention_rate","effective_clean_minority_ratio","mislabeled_retained_ratio","mislabeled_minority_retained_ratio"]].mean().reset_index()
    bm.to_csv(out_dir/"pmlb_mechanism_binary_multiclass_summary.csv", index=False)
    display(summary)
    return df


In [ ]:

# =========================
# MODELS + MITIGATION
# =========================
def detect_cols(X):
    numeric = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]; categorical = [c for c in X.columns if c not in numeric]; return numeric, categorical

def make_preprocessor(X, scale=False):
    numeric, categorical = detect_cols(X)
    num_pipe = Pipeline([("impute", SimpleImputer(strategy="mean"))] + ([("scale", StandardScaler())] if scale else []))
    try: ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError: ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_pipe = Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", ohe)])
    trans = []
    if numeric: trans.append(("num", num_pipe, numeric))
    if categorical: trans.append(("cat", cat_pipe, categorical))
    return ColumnTransformer(trans, remainder="drop")

def get_model(model_name, X, seed, n_classes, mitigation="none"):
    cw = "balanced" if mitigation == "class_weight" else None
    pre = make_preprocessor(X, scale=(model_name=="LogisticRegression"))
    if model_name == "LogisticRegression": clf = LogisticRegression(max_iter=2000, random_state=seed, n_jobs=-1, class_weight=cw)
    elif model_name == "RandomForest": clf = RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1, class_weight=cw)
    elif model_name == "XGBoost":
        if n_classes == 2: clf = XGBClassifier(objective="binary:logistic", eval_metric="logloss", n_estimators=220, learning_rate=0.05, max_depth=6, subsample=0.9, colsample_bytree=0.9, tree_method="hist", random_state=seed, n_jobs=-1)
        else: clf = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", num_class=n_classes, n_estimators=220, learning_rate=0.05, max_depth=6, subsample=0.9, colsample_bytree=0.9, tree_method="hist", random_state=seed, n_jobs=-1)
    elif model_name == "LightGBM": clf = LGBMClassifier(objective="binary" if n_classes==2 else "multiclass", n_estimators=220, learning_rate=0.05, num_leaves=31, subsample=0.9, colsample_bytree=0.9, random_state=seed, n_jobs=-1, verbose=-1, class_weight=cw)
    else: raise ValueError(model_name)
    return Pipeline([("pre", pre), ("clf", clf)])

def random_oversample_training(X, y, seed):
    rng = np.random.default_rng(seed); X = X.reset_index(drop=True); y = pd.Series(y).reset_index(drop=True)
    counts = y.value_counts(); max_count = counts.max(); idx_all = []
    for cls, count in counts.items():
        cls_idx = y[y == cls].index.to_numpy()
        if count < max_count: cls_idx = np.concatenate([cls_idx, rng.choice(cls_idx, size=max_count-count, replace=True)])
        idx_all.extend(cls_idx.tolist())
    idx_all = np.array(idx_all); rng.shuffle(idx_all)
    return X.iloc[idx_all].reset_index(drop=True), y.iloc[idx_all].reset_index(drop=True)

def compute_metrics(model, X_test, y_test):
    pred = model.predict(X_test); labels = np.unique(y_test)
    out = {"accuracy":accuracy_score(y_test,pred), "balanced_accuracy":balanced_accuracy_score(y_test,pred), "macro_f1":f1_score(y_test,pred,average="macro",zero_division=0), "weighted_f1":f1_score(y_test,pred,average="weighted",zero_division=0), "mcc":matthews_corrcoef(y_test,pred)}
    minority = pd.Series(y_test).value_counts().idxmin(); rec = recall_score(y_test,pred,average=None,labels=labels,zero_division=0)
    out["minority_recall"] = float(rec[list(labels).index(minority)]) if minority in labels else np.nan
    return {k:float(v) for k,v in out.items()}

def fit_model(model, X, y, mitigation):
    if mitigation == "class_weight" and model.named_steps["clf"].__class__.__name__ == "XGBClassifier":
        sw = compute_sample_weight(class_weight="balanced", y=y)
        return model.fit(X, y, clf__sample_weight=sw)
    return model.fit(X, y)


In [ ]:

# =========================
# MITIGATION MINI-STUDY
# =========================
def run_mitigation_study():
    out_dir = ensure_dir(Path(OUT_ROOT)/"mitigation"); raw_path = out_dir/"mitigation_raw_results.csv"; fail_path = out_dir/"mitigation_failed_runs.csv"
    done = set(pd.read_csv(raw_path)["run_key"].astype(str)) if raw_path.exists() else set()
    for dataset_name in PMLB_DATASETS:
        print("Mitigation:", dataset_name)
        try: X, y, meta = load_pmlb_dataset(dataset_name)
        except Exception as e: save_row({"dataset_name":dataset_name,"stage":"load","error":str(e)[:500]}, fail_path); continue
        n_classes = int(y.nunique())
        for seed in SEEDS:
            try: Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=seed, stratify=y)
            except Exception as e: save_row({"dataset_name":dataset_name,"seed":seed,"stage":"split","error":str(e)[:500]}, fail_path); continue
            for sid, (sname, noise, missing, imb) in enumerate(MITIGATION_SETTINGS):
                try:
                    Xc, yc, Xtc, ytc, keep, flip, diag = corrupt_train_test(Xtr, ytr, Xte, yte, noise, missing, imb, 900000+seed*1000+sid)
                    if pd.Series(yc).nunique() < 2: raise ValueError("less than two classes")
                except Exception as e: save_row({"dataset_name":dataset_name,"seed":seed,"setting_name":sname,"stage":"corruption","error":str(e)[:500]}, fail_path); continue
                for model_name in MITIGATION_MODELS:
                    for mitigation in MITIGATION_METHODS:
                        run_key = f"{dataset_name}__seed{seed}__{sname}__{model_name}__{mitigation}"
                        if run_key in done: continue
                        try:
                            if mitigation == "random_oversample": Xfit, yfit = random_oversample_training(Xc, yc, seed+999); model_mit = "none"
                            else: Xfit, yfit = Xc, yc; model_mit = mitigation
                            model = get_model(model_name, Xfit, seed, n_classes, mitigation=model_mit)
                            t0 = time.time(); model = fit_model(model, Xfit, yfit, model_mit); metrics = compute_metrics(model, Xtc, ytc)
                            row = {"run_key":run_key,"dataset_name":dataset_name,"seed":seed,"setting_name":sname,"label_noise":noise,"missing_rate":missing,"imbalance_level":str(imb),"model_name":model_name,"mitigation":mitigation,"fit_eval_seconds":round(time.time()-t0,4)}
                            row.update(diag); row.update(metrics); save_row(row, raw_path); done.add(run_key)
                            if len(done) % 100 == 0: print("Completed", len(done))
                        except Exception as e: save_row({"run_key":run_key,"dataset_name":dataset_name,"seed":seed,"setting_name":sname,"model_name":model_name,"mitigation":mitigation,"stage":"fit_eval","error":str(e)[:500]}, fail_path)
    return raw_path

def analyze_mitigation(raw_path):
    out_dir = ensure_dir(Path(OUT_ROOT)/"mitigation"); df = pd.read_csv(raw_path); rows = []
    for (dataset, model, seed, mitigation), g in df.groupby(["dataset_name","model_name","seed","mitigation"]):
        clean = g[g.setting_name == "clean"]
        if len(clean) == 0: continue
        p0 = float(clean.iloc[0].macro_f1)
        for _, r in g.iterrows(): rows.append({"dataset_name":dataset,"model_name":model,"seed":seed,"mitigation":mitigation,"setting_name":r.setting_name,"clean_macro_f1":p0,"macro_f1":float(r.macro_f1),"macro_f1_drop":p0-float(r.macro_f1)})
    drop = pd.DataFrame(rows); drop.to_csv(out_dir/"mitigation_macro_f1_drops.csv", index=False)
    summary = drop.groupby(["mitigation","setting_name"])["macro_f1_drop"].mean().reset_index(); summary.to_csv(out_dir/"mitigation_macro_f1_drop_summary.csv", index=False)
    inter_rows=[]
    for (dataset, model, seed, mitigation), g in drop.groupby(["dataset_name","model_name","seed","mitigation"]):
        vals = {r.setting_name:r.macro_f1_drop for _, r in g.iterrows()}
        if all(k in vals for k in ["noise_only","imbalance_only","noise_imbalance"]): inter_rows.append({"combination":"noise_imbalance","dataset_name":dataset,"model_name":model,"seed":seed,"mitigation":mitigation,"interaction":vals["noise_imbalance"]-(vals["noise_only"]+vals["imbalance_only"])})
        if all(k in vals for k in ["noise_only","missing_only","imbalance_only","all_three"]): inter_rows.append({"combination":"all_three_excess_over_singles","dataset_name":dataset,"model_name":model,"seed":seed,"mitigation":mitigation,"interaction":vals["all_three"]-(vals["noise_only"]+vals["missing_only"]+vals["imbalance_only"])})
    inter = pd.DataFrame(inter_rows); inter.to_csv(out_dir/"mitigation_interactions.csv", index=False)
    inter_summary = inter.groupby(["mitigation","combination"])["interaction"].mean().reset_index(); inter_summary.to_csv(out_dir/"mitigation_interaction_summary.csv", index=False)
    display(summary); display(inter_summary)
    return summary, inter_summary


In [ ]:

# =========================
# OPTIONAL RAW RESULTS STATS
# =========================
def find_raw_csv():
    pats = ["/kaggle/input/**/raw_results_final_24datasets_9models.csv", "/kaggle/input/**/raw_results_final_24datasets_complete_models.csv", "/kaggle/input/**/raw_results_final_unique.csv", "/kaggle/working/**/raw_results_final_24datasets_9models.csv", "/kaggle/working/**/raw_results_final_unique.csv"]
    files=[]
    for p in pats: files += glob.glob(p, recursive=True)
    return list(dict.fromkeys(files))[0] if files else None

def metric_value(df, dataset, model, seed, ln, mr, ir, metric):
    q = df[(df.dataset_name==dataset)&(df.model_name==model)&(df.seed==seed)&(df.label_noise.astype(float)==float(ln))&(df.missing_rate.astype(float)==float(mr))&(df.imbalance_level.astype(str)==str(ir))]
    return np.nan if len(q)==0 else float(q.iloc[0][metric])

def logit_clip(x):
    x = np.clip(float(x), 1e-5, 1-1e-5); return np.log(x/(1-x))

def compute_interactions(raw, metric="macro_f1", transform=None):
    df = raw.copy(); df["imbalance_level"] = df["imbalance_level"].astype(str); out=[]
    noise_levels = sorted([x for x in df.label_noise.astype(float).unique() if x>0]); missing_levels = sorted([x for x in df.missing_rate.astype(float).unique() if x>0]); imb_levels = sorted([x for x in df.imbalance_level.unique() if x!="natural"])
    def getv(d,m,s,ln,mr,ir):
        v = metric_value(df,d,m,s,ln,mr,ir,metric); return np.nan if pd.isna(v) else (transform(v) if transform else v)
    for d in sorted(df.dataset_name.unique()):
      for m in sorted(df.model_name.unique()):
       for s in sorted(df.seed.unique()):
        p0 = getv(d,m,s,0,0,"natural")
        if pd.isna(p0): continue
        for ln in noise_levels:
         for ir in imb_levels:
          pn, pi, pni = getv(d,m,s,ln,0,"natural"), getv(d,m,s,0,0,ir), getv(d,m,s,ln,0,ir)
          if not any(pd.isna(v) for v in [pn,pi,pni]):
           dn, di, dni = p0-pn, p0-pi, p0-pni
           out.append({"combination":"noise_imbalance","dataset_name":d,"model_name":m,"seed":s,"interaction":dni-(dn+di)})
        for ln in noise_levels:
         for mr in missing_levels:
          pn, pm, pnm = getv(d,m,s,ln,0,"natural"), getv(d,m,s,0,mr,"natural"), getv(d,m,s,ln,mr,"natural")
          if not any(pd.isna(v) for v in [pn,pm,pnm]):
           dn, dm, dnm = p0-pn, p0-pm, p0-pnm
           out.append({"combination":"noise_missing","dataset_name":d,"model_name":m,"seed":s,"interaction":dnm-(dn+dm)})
        for mr in missing_levels:
         for ir in imb_levels:
          pm, pi, pmi = getv(d,m,s,0,mr,"natural"), getv(d,m,s,0,0,ir), getv(d,m,s,0,mr,ir)
          if not any(pd.isna(v) for v in [pm,pi,pmi]):
           dm, di, dmi = p0-pm, p0-pi, p0-pmi
           out.append({"combination":"missing_imbalance","dataset_name":d,"model_name":m,"seed":s,"interaction":dmi-(dm+di)})
    return pd.DataFrame(out)

def run_raw_statistical_analysis():
    out_dir = ensure_dir(Path(OUT_ROOT)/"statistical_analysis"); path = find_raw_csv()
    if path is None:
        print("No main raw CSV found; skipping raw-result statistics."); return None
    print("Using raw CSV", path); raw = pd.read_csv(path); raw.to_csv(out_dir/"raw_main_result_copy.csv", index=False)
    raw_inter = compute_interactions(raw, "macro_f1", None); logit_inter = compute_interactions(raw, "macro_f1", logit_clip)
    raw_inter.to_csv(out_dir/"pmlb_interactions_macro_f1_raw_scale.csv", index=False); logit_inter.to_csv(out_dir/"pmlb_interactions_macro_f1_logit_scale.csv", index=False)
    rows=[]
    for scale, inter in [("raw",raw_inter),("logit",logit_inter)]:
        for combo,g in inter.groupby("combination"):
            rows.append({"scale":scale,"combination":combo,"row_equal_mean":g.interaction.mean(),"dataset_equal_mean":g.groupby("dataset_name").interaction.mean().mean(),"dataset_model_equal_mean":g.groupby(["dataset_name","model_name"]).interaction.mean().mean(),"n_rows":len(g),"n_datasets":g.dataset_name.nunique()})
    summ = pd.DataFrame(rows); summ.to_csv(out_dir/"interaction_weighting_summary.csv", index=False); display(summ)
    try:
        fit = smf.mixedlm("interaction ~ C(combination)", raw_inter, groups=raw_inter["dataset_name"]).fit(reml=False, method="lbfgs", maxiter=200)
        (out_dir/"mixedlm_interaction_by_combination.txt").write_text(str(fit.summary()))
        print(fit.summary())
    except Exception as e: (out_dir/"mixedlm_failed.txt").write_text(str(e)); print("MixedLM failed", e)
    try:
        ols = smf.ols("interaction ~ C(combination) + C(model_name)", raw_inter).fit(cov_type="cluster", cov_kwds={"groups":raw_inter.dataset_name})
        (out_dir/"ols_cluster_robust_interaction_model.txt").write_text(str(ols.summary()))
        print(ols.summary())
    except Exception as e: (out_dir/"ols_cluster_robust_failed.txt").write_text(str(e)); print("OLS cluster failed", e)
    return raw_inter


In [ ]:

# =========================
# REPRODUCIBILITY SKELETON
# =========================
def write_repro_skeleton():
    repo = ensure_dir(Path(OUT_ROOT)/"reproducibility_package_skeleton")
    for sub in ["code","data_manifest","raw_results","processed_results","figures","environment"]: ensure_dir(repo/sub)
    (repo/"README.md").write_text("""# Compound Corruption Tabular ML Benchmark\n\nRepository skeleton for manuscript reproducibility.\n\nRequired before submission:\n- exact PMLB and OpenML manifests\n- raw run-level results\n- scripts for all analyses, figures, and tables\n- requirements/environment lockfile\n- instructions for main benchmark and validation studies\n""")
    (repo/"environment"/"requirements.txt").write_text("numpy\npandas\nscikit-learn\npmlb\nxgboost\nlightgbm\nstatsmodels\nmatplotlib\n")
    versions = [f"python={sys.version}", f"numpy={np.__version__}", f"pandas={pd.__version__}"]
    (repo/"environment"/"package_versions.txt").write_text("\n".join(versions))
    for folder in ["data_manifest","mechanism_pmlb","mitigation","statistical_analysis"]:
        src = Path(OUT_ROOT)/folder
        if src.exists():
            dst = ensure_dir(repo/("data_manifest" if folder=="data_manifest" else "processed_results")/folder)
            for f in src.glob("*"):
                if f.is_file(): shutil.copy(f, dst/f.name)
    return repo


In [ ]:

# =========================
# RUN ALL
# =========================
start = time.time()
print("STEP 1: Manifest")
manifest = build_pmlb_manifest()
print("STEP 2: PMLB mechanism diagnostics")
mech = run_pmlb_mechanism_diagnostics()
print("STEP 3: Optional mixed/statistical analysis from raw result CSV")
raw_inter = run_raw_statistical_analysis()
if RUN_MITIGATION:
    print("STEP 4: Mitigation mini-study")
    mraw = run_mitigation_study()
    analyze_mitigation(mraw)
print("STEP 5: Reproducibility skeleton")
write_repro_skeleton()
print("STEP 6: Zip outputs")
zip_path = "/kaggle/working/v10_q1_defense_outputs.zip"
if Path(zip_path).exists(): Path(zip_path).unlink()
shutil.make_archive("/kaggle/working/v10_q1_defense_outputs", "zip", OUT_ROOT)
print("Final ZIP:", zip_path)
print("Runtime hours:", round((time.time()-start)/3600, 3))
